# Detección de Neumonía en Radiografías de Tórax

**Trabajo Final — Especialización en Deep Learning**  
**Autor:** Renzo Ravelli

Este notebook implementa y compara tres arquitecturas de redes neuronales convolucionales para clasificación binaria de radiografías pediátricas de tórax (NORMAL vs PNEUMONIA):

1. **CNN custom** (baseline entrenada desde cero).
2. **ResNet50** con transfer learning desde ImageNet.
3. **EfficientNet-B0** con transfer learning desde ImageNet.

Se prioriza **recall** (sensibilidad) por el contexto clínico: es preferible un falso positivo que dejar pasar un caso real de neumonía.

## 1. Setup y dependencias

In [ ]:
# Clonar el repositorio para tener acceso a src/ en Colab
import os, sys
REPO_NAME = 'DL-Final-Ravelli-Renzo'
GITHUB_USER = 'Senzouz'
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}
sys.path.insert(0, os.getcwd())

In [ ]:
!pip install -q kaggle tqdm scikit-learn seaborn

In [ ]:
# Subir kaggle.json (Kaggle → Account → Create New API Token)
from google.colab import files
import os, shutil, pathlib
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Sube tu kaggle.json:')
    uploaded = files.upload()
    pathlib.Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials OK')

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Descarga del dataset

In [ ]:
from pathlib import Path
from src.data import download_dataset, get_dataloaders, class_distribution, CLASS_NAMES
DATA_ROOT = Path('/content/chest_xray')
download_dataset(DATA_ROOT)
!ls {DATA_ROOT}

## 3. Análisis exploratorio (EDA)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

for split in ['train', 'val', 'test']:
    counts = {c: len(list((DATA_ROOT/split/c).glob('*.jpeg'))) for c in CLASS_NAMES}
    print(f'{split:5s} | NORMAL={counts["NORMAL"]:4d} | PNEUMONIA={counts["PNEUMONIA"]:4d}')

In [ ]:
# Mostrar muestras por clase
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, cls in enumerate(CLASS_NAMES):
    samples = list((DATA_ROOT/'train'/cls).glob('*.jpeg'))[:4]
    for j, p in enumerate(samples):
        axes[i, j].imshow(Image.open(p), cmap='gray')
        axes[i, j].set_title(cls); axes[i, j].axis('off')
plt.tight_layout(); plt.savefig('figures/eda_samples.png', dpi=120); plt.show()

**Observaciones:**
* La validación original solo trae 16 imágenes → re-split desde train con estratificación.
* Dataset desbalanceado (~3:1 a favor de PNEUMONIA) → se usará `class_weight` en la pérdida.

## 4. Preprocesamiento y DataLoaders

In [ ]:
BATCH = 32
IMG = 224
train_loader, val_loader, test_loader, class_weights = get_dataloaders(
    DATA_ROOT, img_size=IMG, batch_size=BATCH, val_frac=0.15, num_workers=2, seed=SEED
)
print(f'Batches → train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}')
print('Class weights:', class_weights)
print('Train dist:', class_distribution(train_loader))

## 5. Modelo 1 — CNN custom (baseline)

In [ ]:
from src.models import build_cnn_custom, build_resnet50, build_efficientnet, unfreeze_last_layers, count_params
from src.train import train_model

cnn = build_cnn_custom()
print('Params (total, trainable):', count_params(cnn))
cnn, hist_cnn = train_model(
    cnn, train_loader, val_loader, device=DEVICE,
    epochs=20, lr=1e-3, class_weights=class_weights, patience=5,
)

## 6. Modelo 2 — ResNet50 (transfer learning)

Dos fases: **feature extraction** con backbone congelado, luego **fine-tuning** del último bloque con LR pequeño.

In [ ]:
resnet = build_resnet50(freeze_backbone=True)
print('Fase 1 — feature extraction')
resnet, hist_res1 = train_model(
    resnet, train_loader, val_loader, device=DEVICE,
    epochs=8, lr=1e-3, class_weights=class_weights, patience=4,
)
print('Fase 2 — fine-tuning')
unfreeze_last_layers(resnet, 'resnet50')
resnet, hist_res2 = train_model(
    resnet, train_loader, val_loader, device=DEVICE,
    epochs=8, lr=1e-5, class_weights=class_weights, patience=4,
)
# Combinar history
from src.train import History
hist_resnet = History(
    train_loss=hist_res1.train_loss + hist_res2.train_loss,
    train_acc=hist_res1.train_acc + hist_res2.train_acc,
    val_loss=hist_res1.val_loss + hist_res2.val_loss,
    val_acc=hist_res1.val_acc + hist_res2.val_acc,
    val_auc=hist_res1.val_auc + hist_res2.val_auc,
)

## 7. Modelo 3 — EfficientNet-B0 (transfer learning)

In [ ]:
effnet = build_efficientnet(freeze_backbone=True)
print('Fase 1 — feature extraction')
effnet, hist_eff1 = train_model(
    effnet, train_loader, val_loader, device=DEVICE,
    epochs=8, lr=1e-3, class_weights=class_weights, patience=4,
)
print('Fase 2 — fine-tuning')
unfreeze_last_layers(effnet, 'efficientnet', num_blocks=2)
effnet, hist_eff2 = train_model(
    effnet, train_loader, val_loader, device=DEVICE,
    epochs=8, lr=1e-5, class_weights=class_weights, patience=4,
)
hist_effnet = History(
    train_loss=hist_eff1.train_loss + hist_eff2.train_loss,
    train_acc=hist_eff1.train_acc + hist_eff2.train_acc,
    val_loss=hist_eff1.val_loss + hist_eff2.val_loss,
    val_acc=hist_eff1.val_acc + hist_eff2.val_acc,
    val_auc=hist_eff1.val_auc + hist_eff2.val_auc,
)

## 8. Curvas de entrenamiento

In [ ]:
from src.evaluate import plot_history
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for i, (name, h) in enumerate([('CNN custom', hist_cnn), ('ResNet50', hist_resnet), ('EfficientNet-B0', hist_effnet)]):
    plot_history(axes[i, 0], axes[i, 1], h, name)
plt.tight_layout(); plt.savefig('figures/training_curves.png', dpi=120); plt.show()

## 9. Evaluación comparativa en test

In [ ]:
import pandas as pd
from src.evaluate import predict, compute_metrics, plot_confusion_matrix, plot_roc_curves

models_dict = {'CNN custom': cnn, 'ResNet50': resnet, 'EfficientNet-B0': effnet}
results, rows = {}, []
for name, m in models_dict.items():
    probs, preds, targets = predict(m, test_loader, DEVICE)
    metrics = compute_metrics(targets, preds, probs)
    results[name] = {'y_true': targets, 'y_pred': preds, 'y_prob': probs, **metrics}
    rows.append({'modelo': name, **metrics})

df = pd.DataFrame(rows).set_index('modelo').round(4)
df.to_csv('results/metrics_comparison.csv')
df

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for i, (name, r) in enumerate(results.items()):
    plot_confusion_matrix(axes[i], r['y_true'], r['y_pred'], name)
plot_roc_curves(axes[3], results)
plt.tight_layout(); plt.savefig('figures/eval_comparison.png', dpi=120); plt.show()

## 10. Interpretabilidad — Grad-CAM

Visualizamos las regiones que activan la predicción del **mejor modelo** según AUC-ROC.

In [ ]:
from src.evaluate import GradCAM, overlay_cam

best_name = df['auc_roc'].idxmax()
best_model = models_dict[best_name].to(DEVICE)
print('Mejor modelo:', best_name)

if best_name == 'ResNet50':
    target_layer = best_model.layer4[-1]
elif best_name == 'EfficientNet-B0':
    target_layer = best_model.features[-1]
else:
    target_layer = best_model.features[-1]
cam_engine = GradCAM(best_model, target_layer)

# Buscar 2 TP y 2 FN
r = results[best_name]
tp_idx = np.where((r['y_true'] == 1) & (r['y_pred'] == 1))[0][:2]
fn_idx = np.where((r['y_true'] == 1) & (r['y_pred'] == 0))[0][:2]
show_idx = list(tp_idx) + list(fn_idx)

test_dataset = test_loader.dataset
fig, axes = plt.subplots(1, len(show_idx), figsize=(4*len(show_idx), 4))
if len(show_idx) == 1: axes = [axes]
for ax, idx in zip(axes, show_idx):
    img, label = test_dataset[idx]
    cam = cam_engine(img.unsqueeze(0).to(DEVICE), class_idx=1)
    ax.imshow(overlay_cam(img, cam))
    pred = r['y_pred'][idx]
    tag = 'TP' if (label == 1 and pred == 1) else 'FN'
    ax.set_title(f'{tag} | real={CLASS_NAMES[label]} pred={CLASS_NAMES[pred]}')
    ax.axis('off')
plt.tight_layout(); plt.savefig('figures/gradcam.png', dpi=120); plt.show()

## 11. Conclusiones

* **Mejor modelo:** el que muestra mayor AUC-ROC y recall en la tabla anterior. Para uso clínico de triage, el recall es la métrica de mayor prioridad.
* **Transfer learning** supera al baseline de manera consistente, gracias a representaciones pre-entrenadas en ImageNet que capturan bordes, texturas y formas reutilizables aún para imágenes médicas.
* **Grad-CAM** confirma que las regiones de mayor activación caen mayoritariamente sobre el parénquima pulmonar, lo cual es plausible clínicamente.
* **Limitaciones:** dataset proviene de una sola fuente (Guangzhou Women and Children's Medical Center) y se compone de pacientes pediátricos; la generalización a adultos o a otros equipos de rayos X requiere validación adicional.
* **Trabajo futuro:** test-time augmentation, ensembles, calibración de probabilidades, validación cruzada.